# Wild Boar (*Sus scrofa*) Connectivity Pipeline — Canton Aargau
**Goal:** Build a permeability / resistance surface for wild boar from GPS telemetry,
following the framework of Fischer et al. (2024) and Clontz et al. (2021).

**Pipeline overview:**
1. Load & clean telemetry  
2. Compute regularised movement metrics  
3. Classify steps as *in-patch* or *in-matrix* (rule-based + speed filter)  
4. Prepare spatial covariates from SwissTLM3D, barriers, wildlife passages  
5. Fit integrated Step-Selection Function (iSSF) on *in-matrix* steps  
6. Convert iSSF coefficients → resistance surface (Keeley et al. 2016)  
7. Diagnostic plots & summary

**Data sources:**
- Telemetry: `WS_Masterfile_final_with_cultures_Aargau.csv`
- SwissTLM3D 2026 (LV95): land cover, water, roads, buildings
- SwissALTI3D / DHM25: terrain
- Wildlife barriers: `alg_WTKbarrieren_20221025.gpkg` (Aargau)
- Wildlife passages: `Wildtierpassagen.gdb` → layer `N2023_Version_Wildtierpassagen`
- Swiss cantonal boundary: `swissBOUNDARIES3D`

## Setup — Packages & Configuration

In [44]:
if (!require("pacman")) install.packages("pacman")

pacman::p_load(
  # Movement ecology
  amt,          # Step-selection functions, track resampling
  momentuHMM,   # HMM state classification (optional, not used in rule path)
  # Spatial
  terra,        # Raster / vector operations
  sf,           # Simple Features vector operations
  # Data wrangling
  dplyr,        # Data manipulation
  tidyr,        # Data reshaping
  purrr,        # Functional programming (map)
  lubridate,    # Date-time parsing
  # Statistics / modelling
  survival,     # Conditional logistic regression (clogit)
  broom,        # Tidy model output
  # Visualisation
  ggplot2,
  patchwork
)

In [45]:
# =============================================================================
# USER CONFIGURATION — adjust paths before running
# =============================================================================

# --- Input paths -------------------------------------------------------------
TELEMETRY_PATH   <- "../data/raw/WS_Masterfile_final_with_cultures_Aargau.csv"
TLM_PATH         <- "../data/raw/SWISSTLM3D_2026_LV95_LN02.gpkg"
DEM_PATH         <- "../data/raw/dhm25_grid_raster.asc"
BOUNDARIES_PATH  <- "../data/raw/swissBOUNDARIES3D_1_5_LV95_LN02.gpkg"
BARRIERS_PATH    <- "../data/raw/alg_WTKbarrieren_20221025.gpkg"
PASSAGES_GDB     <- "../data/raw/Wildtierpassagen.gdb"
PASSAGES_LAYER   <- "N2023_Version_Wildtierpassagen"

# --- Output paths ------------------------------------------------------------
OUT_DIR  <- "../data/processed/"
TEMP_DIR <- "../data/temp/"
dir.create(OUT_DIR,  recursive = TRUE, showWarnings = FALSE)
dir.create(TEMP_DIR, recursive = TRUE, showWarnings = FALSE)

# --- Spatial settings --------------------------------------------------------
TARGET_CRS   <- "EPSG:2056"   # Swiss LV95 (modern standard)
DEM_CRS      <- "EPSG:21781"  # LV03 — native CRS of DHM25
TARGET_RES_M <- 25            # Raster resolution in metres

# --- iSSF settings -----------------------------------------------------------
N_RANDOM_STEPS <- 10          # Random steps per observed step
KEELEY_C       <- 4           # Exponential transformation constant (Fischer 2024)

# --- Telemetry settings ------------------------------------------------------
FIX_RATE_MIN  <- 15           # Nominal GPS fix interval (minutes)
FIX_TOL_MIN   <- 5            # Tolerance around fix interval (minutes)
MIN_DAYS      <- 30           # Minimum tracking duration to include individual

cat("Configuration loaded.\n")

Configuration loaded.


## Step 1 — Load & Clean Telemetry

Load the raw CSV, parse timestamps, and filter out invalid fixes.
Retain only individuals with at least `MIN_DAYS` of tracking data.

In [46]:
cat("── 1. Loading telemetry ──\n")

raw <- read.csv(TELEMETRY_PATH, sep = ";", stringsAsFactors = FALSE,
                fileEncoding = "UTF-8")

tel <- raw %>%
  mutate(
    # Parse timestamp from two separate columns (date + time, UTC)
    datetime_utc = dmy_hms(paste(DatumUTC, ZeitUTC), tz = "UTC"),
    id           = Tier,
    # Convert speed column — already in m per fix-interval in your data
    speed_m      = as.numeric(distance)   # metres per 15-min step
  ) %>%
  filter(
    !is.na(datetime_utc),
    !is.na(X), !is.na(Y),
    X > 400000, X < 900000,   # rough Swiss LV95 bounding box
    Y > 50000,  Y < 350000
  ) %>%
  arrange(id, datetime_utc)

# Retain only individuals with enough data
individual_summary <- tel %>%
  group_by(id) %>%
  summarise(
    n_days  = as.numeric(difftime(max(datetime_utc), min(datetime_utc),
                                  units = "days")),
    n_fixes = n(),
    .groups = "drop"
  ) %>%
  filter(n_days >= MIN_DAYS, n_fixes >= 100)

tel <- tel %>% filter(id %in% individual_summary$id)

cat(sprintf("  Individuals retained : %d\n", n_distinct(tel$id)))
cat(sprintf("  Total GPS fixes      : %d\n", nrow(tel)))

── 1. Loading telemetry ──
  Individuals retained : 14
  Total GPS fixes      : 168236


## Step 2 — Compute Regularised Movement Metrics

Resample each individual's track to the nominal 15-minute fix interval,
then compute step lengths and turning angles via `amt::steps_by_burst()`.
Re-attach the original telemetry covariates (crop type, forest distance,
diel period) needed for the rule-based classification in Step 3.

In [47]:
cat("── 2. Computing movement metrics ──\n")

# Build amt track in Swiss LV95
trk <- tel %>%
  make_track(X, Y, datetime_utc, id = id, crs = TARGET_CRS)

# Resample to FIX_RATE_MIN ± FIX_TOL_MIN
trk_resampled <- trk %>%
  nest(data = -id) %>%
  mutate(data = map(data, function(d) {
    tryCatch(
      track_resample(d,
                     rate      = minutes(FIX_RATE_MIN),
                     tolerance = minutes(FIX_TOL_MIN)),
      error = function(e) NULL
    )
  })) %>%
  filter(!map_lgl(data, is.null)) %>%
  unnest(data)

# Compute steps (step length sl_, turning angle ta_)
steps_raw <- trk_resampled %>%
  nest(data = -id) %>%
  mutate(steps = map(data, steps_by_burst)) %>%
  select(id, steps) %>%
  unnest(steps)

# Re-attach classification covariates from original telemetry
# Join on animal ID + step START time (t1_)
covariates_to_join <- tel %>%
  select(id, datetime_utc, distance, speed_m, Frucht, Walddist, day)

steps_all <- steps_raw %>%
  left_join(covariates_to_join,
            by = c("id" = "id", "t1_" = "datetime_utc")) %>%
  mutate(
    # Compute hour-of-day (0-23) from step start time for diel covariate
    hour = as.integer(format(t1_, "%H")),
    # Remove zero-length steps: turning angle is undefined, confounds HMM
    valid_step = sl_ > 0 & !is.na(sl_)
  )

cat(sprintf("  Total steps computed       : %d\n", nrow(steps_all)))
cat(sprintf("  Steps with zero length     : %d\n", sum(!steps_all$valid_step, na.rm=TRUE)))
cat(sprintf("  Step length distribution   : Q25=%.0fm  Q50=%.0fm  Q75=%.0fm  Q90=%.0fm\n",
            quantile(steps_all$sl_, 0.25, na.rm=TRUE),
            quantile(steps_all$sl_, 0.50, na.rm=TRUE),
            quantile(steps_all$sl_, 0.75, na.rm=TRUE),
            quantile(steps_all$sl_, 0.90, na.rm=TRUE)))

── 2. Computing movement metrics ──
  Total steps computed       : 139390
  Steps with zero length     : 947
  Step length distribution   : Q25=6m  Q50=13m  Q75=41m  Q90=127m


## Step 3 — Classify Steps as *In-Patch* vs *In-Matrix*

### Rationale

Wild boar alternate between two broad movement modes:

| Mode | Biology | Signal in GPS data |
|---|---|---|
| **In-patch** | Resting, wallowing, foraging within a habitat patch | Short steps, high turning angles, low speed |
| **In-matrix** | Directed transit between patches (dispersal, exploration) | Long steps, low turning angles, high speed |

### Classification scheme

We use a **hierarchical rule-based classifier** with thresholds drawn from
published wild boar movement studies:

| Priority | Rule | Threshold | Source |
|---|---|---|---|
| 1 | Speed = 0 | `speed_m == 0` → in-patch | logical (stationary) |
| 2 | High-speed transit | `sl_ >= 250 m` → in-matrix | Podgórski et al. 2013 |
| 3 | Directed movement | `sl_ >= 150 m` AND `abs(ta_) < π/4` → in-matrix | Clontz et al. 2021 |
| 4 | Daytime resting in forest | `day == "Day"` AND `Walddist == 0` AND `sl_ < 50 m` → in-patch | Fischer & Ranzoni 2017 |
| 5 | Nocturnal field foraging | `day == "Night"` AND crop in foraging list AND `sl_ < 150 m` → in-patch | Schley & Roper 2003 |
| 6 | Nocturnal open transit | `day == "Night"` AND `Walddist > 50 m` AND `sl_ >= 50 m` → in-matrix | Thurfjell et al. 2009 |
| 7 | Default | everything else → in-patch | conservative prior |

**Threshold justification:**
- **250 m** at 15-min fix = 1,000 m/h = clearly transit (Podgórski et al. 2013 report mean
  transit steps of 398 ± 386 m for males).
- **150 m + directed** combines distance and directionality (Clontz et al. 2021 travelling
  state mean: females 244 m, males 398 m; turning angle ≈ 0).
- **50 m** upper bound for resting/foraging follows Clontz et al. 2021 (resting mean ~11 m,
  foraging mean ~38 m).

In [48]:
# Crop species known to attract nocturnal wild boar foraging (Schley & Roper 2003)
FORAGING_CROPS <- c("Mais", "Weizen", "Erbsen", "Kartoffeln",
                     "Rueben", "Raps", "Gerste", "Sonnenblumen",
                     "Triticale", "Hafer")

steps_classified <- steps_all %>%
  mutate(
    state_label = case_when(

      # --- RULE 1: Stationary fix — definitively in-patch ----------------------
      # Speed = 0 means the GPS recorded an identical or near-identical position.
      # No transit movement is possible. Source: logical constraint.
      !valid_step | speed_m == 0
        ~ "in-patch",

      # --- RULE 2: High-speed long-distance transit ----------------------------
      # Steps >= 250 m in 15 min exceed the foraging movement budget of wild boar
      # at any time of day. These are pure matrix-crossing movements.
      # Source: Podgórski et al. 2013; mean male transit step ~398 m.
      sl_ >= 250
        ~ "in-matrix",

      # --- RULE 3: Directed medium-distance movement ---------------------------
      # Steps >= 150 m with a small turning angle (< 45°) indicate purposeful
      # directional travel, not local foraging loops.
      # Source: Clontz et al. 2021 (travelling state: mean ~244–420 m, ta ≈ 0).
      sl_ >= 150 & !is.na(ta_) & abs(ta_) < (pi / 4)
        ~ "in-matrix",

      # --- RULE 4: Daytime resting in forest -----------------------------------
      # During daylight, wild boar in their core habitat rest in forest cover.
      # Short steps + inside forest = unambiguously in-patch.
      # Source: Fischer & Ranzoni 2017; Keuling et al. 2008.
      tolower(day) == "day" & !is.na(Walddist) & Walddist == 0 & sl_ < 50
        ~ "in-patch",

      # --- RULE 5: Nocturnal field foraging ------------------------------------
      # Slow movement at night inside a known foraging crop = resource patch use.
      # Source: Thurfjell et al. 2009; Schley & Roper 2003.
      tolower(day) == "night" &
      !is.na(Frucht) & Frucht %in% FORAGING_CROPS &
      sl_ < 150
        ~ "in-patch",

      # --- RULE 6: Nocturnal open-area transit ---------------------------------
      # Night movement at moderate speed far from forest cover and not in a
      # foraging crop = transit through risky open matrix habitat.
      # Source: Thurfjell et al. 2009; Podgórski et al. 2013.
      tolower(day) == "night" &
      !is.na(Walddist) & Walddist > 50 &
      (is.na(Frucht) | !Frucht %in% FORAGING_CROPS) &
      sl_ >= 50
        ~ "in-matrix",

      # --- RULE 7: Conservative default ----------------------------------------
      # All remaining ambiguous steps are assigned in-patch.
      # Wild boar spend ~70-80% of their time budget in patch activities
      # (Blasetti et al. 1988), so a conservative default is appropriate.
      TRUE ~ "in-patch"
    )
  )

# Summary
tbl <- table(steps_classified$state_label)
cat("Classification summary:\n")
cat(sprintf("  in-patch  : %d (%.1f%%)\n", tbl["in-patch"],
            100 * tbl["in-patch"] / sum(tbl)))
cat(sprintf("  in-matrix : %d (%.1f%%)\n", tbl["in-matrix"],
            100 * tbl["in-matrix"] / sum(tbl)))

# Save both subsets
in_patch  <- steps_classified %>% filter(state_label == "in-patch")
in_matrix <- steps_classified %>% filter(state_label == "in-matrix")

write.csv(in_patch,  file.path(OUT_DIR, "WildBoar_InPatch_Steps.csv"),  row.names = FALSE)
write.csv(in_matrix, file.path(OUT_DIR, "WildBoar_InMatrix_Steps.csv"), row.names = FALSE)
cat("  Saved: WildBoar_InPatch_Steps.csv and WildBoar_InMatrix_Steps.csv\n")

Classification summary:
  in-patch  : 130002 (93.3%)
  in-matrix : 9388 (6.7%)
  Saved: WildBoar_InPatch_Steps.csv and WildBoar_InMatrix_Steps.csv


## Step 4 — Prepare Spatial Covariates

Build a 25-m raster stack covering Canton Aargau (+ 2 km buffer) from:

| Layer | Source | Wild boar relevance |
|---|---|---|
| Forest cover | SwissTLM3D `tlm_bb_bodenbedeckung` | Resting cover, patch habitat |
| Forest density | Focal sum (r = 50 m) of forest | Stepping-stone quality |
| Distance to forest edge | Derived from forest binary | Edge-foraging behaviour |
| Slope / TRI | DHM25 via terrain() | Energy cost of movement |
| Distance to water & cooling | SwissTLM3D water + wetlands + fens | Thermoregulation, wallowing |
| Distance to infrastructure | Roads, buildings, railways, settlements | Barrier / avoidance |
| Highway binary | Autobahn (fenced) | Absolute barrier |
| Wildlife barriers | `alg_WTKbarrieren_20221025.gpkg` | Connectivity constraint |
| Wildlife passages | `Wildtierpassagen.gdb` | Connectivity enhancement |
| NDVI proxy | Forest density (replace with Sentinel-2 if available) | Vegetation productivity |

In [49]:
cat("── 4a. Study area boundary ──\n")

# Load Swiss cantonal boundaries and extract Aargau
kantone <- st_read(BOUNDARIES_PATH, layer = "tlm_kantonsgebiet", quiet = TRUE) %>%
  st_transform(2056)
aargau_poly <- kantone %>% filter(tolower(name) == "aargau")
aargau_vect <- vect(aargau_poly)

# 2 km buffer to prevent edge effects during distance calculations
aargau_buffer <- st_buffer(aargau_poly, dist = 2000)
aargau_wkt    <- st_as_text(st_geometry(aargau_buffer))  # for st_read wkt_filter

# Master raster grid: 25 m resolution, LV95, snapped to Aargau extent
master_grid <- rast(ext(aargau_vect), resolution = TARGET_RES_M, crs = "EPSG:2056")

cat(sprintf("  Aargau extent: xmin=%.0f xmax=%.0f ymin=%.0f ymax=%.0f\n",
            ext(aargau_vect)[1], ext(aargau_vect)[2],
            ext(aargau_vect)[3], ext(aargau_vect)[4]))
cat(sprintf("  Master grid: %d rows x %d cols\n",
            nrow(master_grid), ncol(master_grid)))

── 4a. Study area boundary ──
  Aargau extent: xmin=2620698 xmax=2676827 ymin=1221173 ymax=1274772
  Master grid: 2144 rows x 2245 cols


In [50]:
cat("── 4b. Forest cover and density ──\n")

# =============================================================================
# 1. LOAD FOREST POLYGONS
# =============================================================================
bodenbedeckung <- st_read(TLM_PATH,
                           layer      = "tlm_bb_bodenbedeckung",
                           wkt_filter = aargau_wkt,
                           quiet      = TRUE)

# Forest classes in SwissTLM3D Bodenbedeckung
FOREST_CLASSES <- c("Wald", "Wald offen", "Gebueschwald", "Gehoelzflaeche")

forest_sf <- bodenbedeckung %>% filter(objektart %in% FOREST_CLASSES)

# =============================================================================
# 2. RASTERIZE BINARY FOREST
# =============================================================================
# Binary forest raster (1 = forest, 0 = non-forest)
# Added touches = TRUE to ensure narrow tree lines (Gehoelzflaeche) are captured!
r_forest <- rasterize(vect(forest_sf), master_grid, field = 1, background = 0, touches = TRUE)
r_forest <- mask(r_forest, aargau_vect)
names(r_forest) <- "forest"

# =============================================================================
# 3. CALCULATE FOREST DENSITY
# =============================================================================
cat("  Calculating 50m forest density...\n")
# Moving window: proportion of forest pixels in a 50m radius
w_50m         <- focalMat(r_forest, 50, "circle")
r_forest_dens <- focal(r_forest, w = w_50m, fun = "mean", na.rm = TRUE)
r_forest_dens <- mask(r_forest_dens, aargau_vect)
names(r_forest_dens) <- "forest_density"

# =============================================================================
# 4. CALCULATE DISTANCE TO NEAREST FOREST
# =============================================================================
cat("  Calculating continuous distance to forest...\n")
# Step A: Isolate the forest pixels. distance() needs the background to be NA, not 0!
r_forest_only <- ifel(r_forest == 1, 1, NA)

# Step B: Calculate true Euclidean distance in meters
r_dist_forest <- distance(r_forest_only)
r_dist_forest <- mask(r_dist_forest, aargau_vect)
names(r_dist_forest) <- "dist_forest_edge"

# =============================================================================
# 5. SAVE OUTPUTS
# =============================================================================
writeRaster(r_forest,       file.path(TEMP_DIR, "forest.tif"),           overwrite=TRUE)
writeRaster(r_forest_dens,  file.path(TEMP_DIR, "forest_density.tif"),   overwrite=TRUE)
writeRaster(r_dist_forest,  file.path(TEMP_DIR, "dist_forest_edge.tif"), overwrite=TRUE)

cat("  Forest layers saved.\n")

── 4b. Forest cover and density ──
  Calculating 50m forest density...
  Calculating continuous distance to forest...
  Forest layers saved.


In [51]:
cat("── 4c. Terrain (slope, TRI) from DHM25 ──\n")

# DHM25 ships in LV03 (EPSG:21781) without embedded CRS — assign manually
dem_raw      <- rast(DEM_PATH)
crs(dem_raw) <- DEM_CRS   # "EPSG:21781"

# Transform the buffer polygon to LV03 (fast: transform polygon, not raster)
aargau_buffer_lv03 <- st_transform(aargau_buffer, 21781)

# Crop to Aargau in native projection (reduces data volume before reprojection)
dem_cropped <- crop(dem_raw, ext(aargau_buffer_lv03))

# Reproject the small cropped DEM to LV95 / 25 m master grid
dem_lv95 <- project(dem_cropped, master_grid, method = "bilinear")

# Slope in degrees (energy cost proxy; Fischer et al. 2024 use slope for red deer)
r_slope <- terrain(dem_lv95, v = "slope", unit = "degrees")
r_slope  <- mask(r_slope, aargau_vect)
names(r_slope) <- "slope"

# Terrain Ruggedness Index (Riley et al. 1999)
# Preferred over raw slope for wild boar: captures habitat heterogeneity
r_tri <- terrain(dem_lv95, v = "TRI")
r_tri <- mask(r_tri, aargau_vect)
names(r_tri) <- "tri"

writeRaster(r_slope, file.path(TEMP_DIR, "slope.tif"), overwrite=TRUE)
writeRaster(r_tri,   file.path(TEMP_DIR, "tri.tif"),   overwrite=TRUE)
cat("  Terrain layers saved.\n")

── 4c. Terrain (slope, TRI) from DHM25 ──
  Terrain layers saved.


In [52]:
cat("── 4d. Distance to water ──\n")

# =============================================================================
# 1. EXTRACT WATER POLYGONS
# =============================================================================
# We use exclusively the SwissTLM3D land cover layer (tlm_bb_bodenbedeckung).
# Cooling habitats (wetlands, swamps, riparian zones) are explicitly excluded.

WATER_CLASSES <- c("Fliessgewaesser", "Stehende Gewaesser")

cat("  Filtering pure water bodies from bodenbedeckung...\n")
water_sf <- bodenbedeckung %>%
  filter(objektart %in% WATER_CLASSES)

# =============================================================================
# 2. RASTERIZE WATER BODIES
# =============================================================================
r_water_bin <- rast(master_grid)
r_water_bin[] <- NA

if (nrow(water_sf) > 0) {
  cat("  Rasterizing water polygons (touches = TRUE)...\n")
  # touches = TRUE ensures that narrow rivers are captured reliably 
  # even if they don't cover the absolute center of a 25m pixel.
  r_water_bin <- rasterize(vect(water_sf), r_water_bin, field = 1, 
                           update = TRUE, touches = TRUE)
}

# =============================================================================
# 3. CALCULATE DISTANCE
# =============================================================================
cat("  Calculating continuous distance to water...\n")

# terra calculates true Euclidean distance in CRS units (meters) automatically
r_dist_water <- distance(r_water_bin)

# Mask to the exact shape of the Canton Aargau
r_dist_water <- mask(r_dist_water, aargau_vect)
names(r_dist_water) <- "dist_water"

# =============================================================================
# 4. SAVE OUTPUT
# =============================================================================
writeRaster(r_dist_water, file.path(TEMP_DIR, "dist_water.tif"), overwrite = TRUE)

cat("  Water distance layer saved successfully.\n")

# Optional: Print a quick summary of the coverage
total_pixels <- global(!is.na(r_dist_water), "sum", na.rm = TRUE)[[1]]
water_pixels <- global(r_water_bin, "sum", na.rm = TRUE)[[1]]

if(!is.null(water_pixels)) {
  cat(sprintf("  Water pixels: %d (%.1f%% of Aargau)\n", 
              water_pixels, 
              100 * water_pixels / total_pixels))
}

── 4d. Distance to water ──
  Filtering pure water bodies from bodenbedeckung...
  Rasterizing water polygons (touches = TRUE)...
  Calculating continuous distance to water...
  Water distance layer saved successfully.
  Water pixels: 102065 (4.5% of Aargau)


In [53]:
cat("── 4e/f. Infrastructure & Passages (Unified) ──\n")

# =============================================================================
# 1. LOAD INFRASTRUCTURE (No Buffers, using touches = TRUE)
# =============================================================================
cat("  Loading infrastructure layers...\n")

strassen  <- st_read(TLM_PATH, layer = "tlm_strassen_strasse", wkt_filter = aargau_wkt, quiet = TRUE)
gebaeude  <- st_read(TLM_PATH, layer = "tlm_bauten_gebaeude_footprint", wkt_filter = aargau_wkt, quiet = TRUE)

eisenbahn <- tryCatch(st_read(TLM_PATH, layer = "tlm_oev_eisenbahn", wkt_filter = aargau_wkt, quiet = TRUE), 
                      error = function(e) NULL)
areal     <- tryCatch(st_read(TLM_PATH, layer = "tlm_areale_nutzungsareal", wkt_filter = aargau_wkt, quiet = TRUE), 
                      error = function(e) NULL)

HIGHWAY_CLASSES  <- c("Autobahn", "Autostrasse", "Ausfahrt", "Einfahrt")
MAINROAD_CLASSES <- c("10m Strasse", "8m Strasse", "6m Strasse")

autobahn <- strassen %>% filter(objektart %in% HIGHWAY_CLASSES)
hauptstr <- strassen %>% filter(objektart %in% MAINROAD_CLASSES)

EXCLUDE_AREALS <- c("Wald nicht bestockt", "Abbauareal", "Obstanlage", "Friedhof", "Reben", "Gewässerareal")
siedlung <- if (!is.null(areal)) areal %>% filter(!objektart %in% EXCLUDE_AREALS) else NULL

# =============================================================================
# 2. LOAD PASSAGES (Barriers excluded)
# =============================================================================
cat("  Loading wildlife passages...\n")

passages_sf <- tryCatch(
  st_read(PASSAGES_GDB, layer = PASSAGES_LAYER, quiet = TRUE) %>% 
    st_transform(2056) %>% st_filter(aargau_buffer),
  error = function(e) { message("  Note: passages layer could not be loaded."); NULL }
)

# =============================================================================
# 3. RASTERIZE BASE INFRASTRUCTURE (Distance Calculation)
# =============================================================================
r_infra <- rast(master_grid); r_infra[] <- NA

# Helper function using touches = TRUE instead of st_buffer
stamp <- function(r, sf_obj, val = 1) {
  if (!is.null(sf_obj) && nrow(sf_obj) > 0) {
    rasterize(vect(sf_obj), r, field = val, update = TRUE, touches = TRUE)
  } else {
    r
  }
}

cat("  Rasterizing base infrastructure (touches = TRUE)...\n")
r_infra <- stamp(r_infra, gebaeude)
r_infra <- stamp(r_infra, autobahn)
r_infra <- stamp(r_infra, hauptstr)
r_infra <- stamp(r_infra, eisenbahn)
r_infra <- stamp(r_infra, siedlung)

# Calculate true continuous distance from standard infrastructure
cat("  Calculating continuous distance to infrastructure...\n")
r_dist_infra <- distance(r_infra)

# =============================================================================
# 4. INJECT FIXED EFFECTS (Buffered Passages)
# =============================================================================
# Passages = Safe Zone (Overwrites the road beneath it)
if (!is.null(passages_sf) && nrow(passages_sf) > 0) {
  cat("  Injecting fixed effects: Passages (Buffered Safe Zone)...\n")
  
  # Buffer the passages by 30 meters to ensure the corridor is wide enough 
  # to be captured solidly by the 25m raster grid.
  passages_buf <- st_buffer(passages_sf, dist = 30)
  
  # Create a temporary binary mask of the exact passage polygons
  r_passage_mask <- rast(master_grid); r_passage_mask[] <- NA
  r_passage_mask <- stamp(r_passage_mask, passages_buf)
  
  # Overwrite the distance. A value of 500m simulates being far away from danger.
  # This punches a "safe hole" directly through the 0-value highway pixels in the distance layer!
  SAFE_DISTANCE_VALUE <- 500 
  r_dist_infra[r_passage_mask == 1] <- SAFE_DISTANCE_VALUE
}

# =============================================================================
# 5. MASK AND SAVE
# =============================================================================
r_dist_infra <- mask(r_dist_infra, aargau_vect)
names(r_dist_infra) <- "dist_infra"

writeRaster(r_dist_infra, file.path(TEMP_DIR, "dist_infra.tif"), overwrite = TRUE)

# --- Highway Raster Logic ---
r_highway <- rast(master_grid); r_highway[] <- 0
r_highway <- stamp(r_highway, autobahn)

# PUNCH A HOLE IN THE HIGHWAY BARRIER:
# If a pixel is a buffered passage, it is NO LONGER considered a highway barrier.
# This prevents Step 6 from overwriting our passage with R_max!
if (!is.null(passages_sf) && nrow(passages_sf) > 0) {
  r_highway[r_passage_mask == 1] <- 0
}

r_highway <- mask(r_highway, aargau_vect)
names(r_highway) <- "highway"

writeRaster(r_highway, file.path(TEMP_DIR, "highway.tif"), overwrite = TRUE)

cat("── Infrastructure & Passages unified successfully. ──\n")

── 4e/f. Infrastructure & Passages (Unified) ──
  Loading infrastructure layers...
  Loading wildlife passages...
  Rasterizing base infrastructure (touches = TRUE)...
  Calculating continuous distance to infrastructure...
  Injecting fixed effects: Passages (Buffered Safe Zone)...
── Infrastructure & Passages unified successfully. ──


In [54]:
# cat("── 4f. Wildlife barriers and passages ──\n")

# # --- Wildlife barriers (alg_WTKbarrieren_20221025.gpkg) ----------------------
# # Three severity classes (Stufe I–III) representing increasing impermeability.
# # Classification from the Aargau cantonal authority (ALG):
# #   Stufe I   (15 features): major barrier, e.g. motorway without passage
# #   Stufe II  (29 features): significant barrier, e.g. railway
# #   Stufe III (54 features): moderate barrier, e.g. cantonal road
# #
# # We rasterise each class separately so the iSSF can estimate
# # class-specific deterrence effects, and also create a combined
# # ordinal barrier severity raster (0 = no barrier, 3 = Stufe I).

# barriers_sf <- st_read(BARRIERS_PATH,
#                         layer = "alg_WTKbarrieren_20221025",
#                         quiet = TRUE) %>%
#   st_transform(2056)

# # Buffer lines by severity-appropriate width before rasterising
# # (Stufe I = 30 m corridor of barrier influence, Stufe III = 10 m)
# barrier_widths <- c("Stufe I" = 30, "Stufe II" = 20, "Stufe III" = 10)

# r_barrier_severity <- rast(master_grid); r_barrier_severity[] <- 0

# for (stufe in c("Stufe III", "Stufe II", "Stufe I")) {
#   b_sf  <- barriers_sf %>% filter(Typ == stufe)
#   if (nrow(b_sf) == 0) next
#   b_buf <- st_buffer(b_sf, dist = barrier_widths[stufe])
#   # Higher severity overwrites lower (paint in ascending order)
#   sev_value <- switch(stufe, "Stufe III" = 1, "Stufe II" = 2, "Stufe I" = 3)
#   r_stufe <- rasterize(vect(b_buf), master_grid,
#                         field = sev_value, background = NA)
#   r_barrier_severity <- ifel(!is.na(r_stufe), r_stufe, r_barrier_severity)
# }
# r_barrier_severity <- mask(r_barrier_severity, aargau_vect)
# names(r_barrier_severity) <- "barrier_severity"

# # Distance to nearest barrier of any class
# r_barrier_bin  <- ifel(r_barrier_severity > 0, 1, NA)
# r_dist_barrier <- distance(r_barrier_bin)
# r_dist_barrier <- mask(r_dist_barrier, aargau_vect)
# names(r_dist_barrier) <- "dist_barrier"

# # --- Wildlife passages (Wildtierpassagen.gdb) ---------------------------------
# # Passages reduce effective resistance for wild boar crossing barriers.
# # We create a binary layer marking passage locations and a distance-to-passage
# # raster that can be used to reduce local resistance near crossings.

# passages_sf <- tryCatch(
#   st_read(PASSAGES_GDB, layer = PASSAGES_LAYER, quiet = TRUE) %>%
#     st_transform(2056) %>%
#     # Spatial filter to Aargau buffer
#     st_filter(aargau_buffer),
#   error = function(e) {
#     message("  Note: passages layer could not be loaded — check GDB path/layer name.")
#     NULL
#   }
# )

# if (!is.null(passages_sf) && nrow(passages_sf) > 0) {
#   cat(sprintf("  Passages loaded: %d features\n", nrow(passages_sf)))
#   # Buffer passages by 50 m (zone of reduced resistance)
#   passages_buf  <- st_buffer(passages_sf, dist = 50)
#   r_passage_bin <- rasterize(vect(passages_buf), master_grid,
#                                field = 1, background = NA)
#   r_dist_passage <- distance(r_passage_bin)
#   r_dist_passage <- mask(r_dist_passage, aargau_vect)
#   names(r_dist_passage) <- "dist_passage"

#   writeRaster(r_dist_passage, file.path(TEMP_DIR, "dist_passage.tif"),
#               overwrite = TRUE)
# } else {
#   cat("  Passages not loaded — dist_passage layer skipped.\n")
#   r_dist_passage <- NULL
# }

# writeRaster(r_barrier_severity, file.path(TEMP_DIR, "barrier_severity.tif"),
#             overwrite = TRUE)
# writeRaster(r_dist_barrier,     file.path(TEMP_DIR, "dist_barrier.tif"),
#             overwrite = TRUE)
# cat("  Barrier and passage layers saved.\n")

In [55]:
cat("── 4g. Assemble covariate stack ──\n")

# Collect all raster layers into a named stack for iSSF extraction
# Passage distance is optional — include only if successfully loaded
base_layers <- list(
  forest           = r_forest,
  forest_density   = r_forest_dens,
  dist_forest_edge = r_forest_edge_dist,
  slope            = r_slope,
  tri              = r_tri,
  dist_water       = r_dist_water,
  dist_infra       = r_dist_infra
)

if (!is.null(r_dist_passage)) {
  base_layers[["dist_passage"]] <- r_dist_passage
}

cov_stack        <- rast(base_layers)
names(cov_stack) <- names(base_layers)

cat(sprintf("  Covariate stack: %d layers, res=%d m, CRS=%s\n",
            nlyr(cov_stack), TARGET_RES_M, crs(cov_stack, describe=TRUE)$code))
print(names(cov_stack))

writeRaster(cov_stack, file.path(TEMP_DIR, "covariate_stack.tif"),
            overwrite = TRUE)
cat("  covariate_stack.tif saved.\n")

── 4g. Assemble covariate stack ──
  Covariate stack: 8 layers, res=25 m, CRS=2056
[1] "forest"           "forest_density"   "dist_forest_edge" "slope"            "tri"              "dist_water"      
[7] "dist_infra"       "dist_passage"    
  covariate_stack.tif saved.


## Step 5 — Integrated Step-Selection Function (iSSF)

Fit a conditional logistic regression on the **in-matrix steps only**,
comparing observed steps to 10 random alternatives drawn from the
empirical step-length and turning-angle distributions.

Covariates are z-standardised so that coefficients are directly comparable.
The resulting beta coefficients represent habitat *selection* during transit:
- **Positive β**: habitat preferred during matrix crossing → lower resistance
- **Negative β**: habitat avoided during transit → higher resistance

References: Avgar et al. (2016); Fischer et al. (2024).

In [60]:
cat("── 5. Fitting iSSF on in-matrix steps ──\n")

# --- 1. Prepare clean in-matrix step data ------------------------------------
# Require valid step length, turning angle, and at least one covariate
in_matrix_clean <- in_matrix %>%
  filter(
    !is.na(sl_),
    !is.na(ta_),
    sl_ > 0           # Zero-length steps have undefined turning angles
  )

cat(sprintf("  In-matrix steps for iSSF : %d\n", nrow(in_matrix_clean)))

# Restore amt step class so random_steps() works correctly
class(in_matrix_clean) <- c("steps_xyt", "steps_xy", "data.frame")

# --- 2. Generate random control steps ----------------------------------------
issf_data <- in_matrix_clean %>%
  random_steps(n_control = N_RANDOM_STEPS) %>%
  mutate(
    log_sl = log(sl_ + 1),   # Log-transform step length (movement kernel term)
    cos_ta = cos(ta_)        # Cosine of turning angle (directional persistence)
  )

cat(sprintf("  Total rows (observed + random): %d\n", nrow(issf_data)))

# --- 3. Extract spatial covariates at step endpoints -------------------------
cat("  Extracting spatial covariates at step endpoints...\n")

# Create spatial points from the step endpoints (where the boar moved TO)
# IMPORTANT: Adjust EPSG to 2056 if your raw telemetry is already in LV95!
pts <- terra::vect(issf_data, geom = c("x2_", "y2_"), crs = "EPSG:21781")

# The Magic Fix: Force the points into the exact CRS of the raster maps
pts_matched <- terra::project(pts, terra::crs(cov_stack))

# Extract values (points and rasters now perfectly align)
ext_vals <- terra::extract(cov_stack, pts_matched)

# Bind the extracted values (dropping the first ID column from terra::extract)
issf_data <- cbind(issf_data, ext_vals[, -1])   

# Remove steps where any extracted covariate is NA (fell outside the map)
cov_names <- names(cov_stack)
issf_data <- issf_data %>%
  filter(complete.cases(across(all_of(cov_names)), log_sl, cos_ta))

cat(sprintf("  Valid steps after covariate extraction: %d\n", nrow(issf_data)))

if (nrow(issf_data) == 0) {
  stop("No valid steps after covariate extraction. Check CRS alignment (LV03 vs LV95)!")
}

# --- 4. Validate and standardise continuous covariates -----------------------
cat("  Checking and scaling landscape variables...\n")

# Define the covariates we expect from the raster stack
expected_covs <- c("forest_density", "dist_forest_edge", "slope", "tri",
                   "dist_water", "dist_infra")

# Only keep the variables that actually exist in the dataset
available_covs <- intersect(expected_covs, names(issf_data))

# Filter out "dead" variables to prevent Division by Zero during scaling
valid_covs <- c()
for (cov in available_covs) {
  cov_sd <- sd(issf_data[[cov]], na.rm = TRUE)
  
  if (is.na(cov_sd) || cov_sd == 0) {
    cat(sprintf("  -> WARNING: Raster '%s' is ignored! (Values are constant or missing)\n", cov))
  } else {
    valid_covs <- c(valid_covs, cov)
  }
}

# Calculate and SAVE the scaling parameters (Mean & SD)
# This is CRITICAL so Step 6 can scale the prediction rasters exactly the same way!
scale_params <- issf_data %>%
  summarise(across(all_of(valid_covs),
                   list(mean = ~ mean(.x, na.rm = TRUE),
                        sd   = ~ sd(.x,   na.rm = TRUE))))

# Apply the Z-standardization safely using the verified variables
issf_data <- issf_data %>%
  mutate(across(all_of(valid_covs),
                ~ (.x - mean(.x, na.rm = TRUE)) / sd(.x, na.rm = TRUE),
                .names = "{.col}_sc"))

# Safely convert binary forest to a categorical factor
issf_data$forest_fct <- as.factor(issf_data$forest)

# --- 5. Build formula dynamically --------------------------------------------
# Automatically assemble the formula using only the valid, scaled variables
scaled_covs <- paste0(valid_covs, "_sc")
form_terms  <- c("forest_fct", "log_sl", "cos_ta", scaled_covs, "strata(step_id_)")

frm <- as.formula(paste("case_ ~", paste(form_terms, collapse = " + ")))

cat("  Dynamic formula created:\n"); print(frm)

# Final safety check: drop any lingering hidden NAs
issf_model_data <- issf_data %>% 
  tidyr::drop_na(all_of(c("case_", "forest_fct", "log_sl", "cos_ta", scaled_covs)))

cat(sprintf("  Rows effectively used in model: %d\n", nrow(issf_model_data)))

if (nrow(issf_model_data) == 0) {
  stop("0 rows remaining. Check the 'forest' variable, it might be generating NAs.")
}

# --- 6. Fit conditional logistic regression ----------------------------------
cat("  Fitting clogit (iSSF Model)...\n")

m_issf <- survival::clogit(frm, data = issf_model_data, method = "efron")

cat("  iSSF Model SUCCESSFULLY fitted!\n")
print(summary(m_issf))

# --- 7. Plot and save results ------------------------------------------------
# Extract coefficients (excluding step metrics from the plot)
coef_df <- broom::tidy(m_issf, conf.int = TRUE) %>%
  filter(!grepl("log_sl|cos_ta|step_id", term)) %>%
  arrange(desc(abs(estimate)))

write.csv(coef_df, file.path(OUT_DIR, "issf_coefficients.csv"), row.names = FALSE)

# Generate coefficient plot
p_coef <- ggplot(coef_df,
                 aes(x = estimate, y = reorder(term, estimate),
                     xmin = conf.low, xmax = conf.high)) +
  geom_vline(xintercept = 0, linetype = "dashed", colour = "grey60") +
  geom_pointrange(colour = "#1D6FA5", size = 0.7, linewidth = 0.9) +
  labs(x = "Selection coefficient β (positive = preferred during transit)",
       y = NULL,
       title = "iSSF — Wild Boar In-Matrix Habitat Selection",
       subtitle = "Canton Aargau | Only in-matrix steps") +
  theme_minimal(base_size = 11)

ggsave(file.path(OUT_DIR, "issf_coefficients.png"), p_coef,
       width = 8, height = 6, dpi = 200)
cat("  Saved: issf_coefficients.png\n")

── 5. Fitting iSSF on in-matrix steps ──
  In-matrix steps for iSSF : 9300
  Total rows (observed + random): 102289
  Extracting spatial covariates at step endpoints...
  Valid steps after covariate extraction: 100704
  Checking and scaling landscape variables...
  -> WARNING: Raster 'dist_forest_edge' is ignored! (Values are constant or missing)
  Dynamic formula created:
case_ ~ forest_fct + log_sl + cos_ta + forest_density_sc + slope_sc + 
    tri_sc + dist_water_sc + dist_infra_sc + strata(step_id_)
  Rows effectively used in model: 100704
  Fitting clogit (iSSF Model)...
  iSSF Model SUCCESSFULLY fitted!
Call:
coxph(formula = Surv(rep(1, 100704L), case_) ~ forest_fct + log_sl + 
    cos_ta + forest_density_sc + slope_sc + tri_sc + dist_water_sc + 
    dist_infra_sc + strata(step_id_), data = issf_model_data, 
    method = "efron")

  n= 100704, number of events= 9176 

                      coef exp(coef) se(coef)      z Pr(>|z|)    
forest_fct1       -0.08513   0.91839  0.06897 -

## Step 6 — Resistance Surface (Keeley et al. 2016)

Convert iSSF selection coefficients to a resistance surface following
Fischer et al. (2024) and Keeley et al. (2016):

1. Compute the **linear predictor η** from scaled raster covariates and β
2. Normalise to **habitat suitability S ∈ [0, 1]** (high S = preferred)
3. Apply exponential transformation: **R = exp(C × (1 − S))**,
   with C = 4 giving R ∈ [1, 54.6]
4. Set absolute barriers (highways, Stufe-I barriers, open water) to R_max

In [64]:
# =============================================================================
# STEP 6: CONVERT COEFFICIENTS → RESISTANCE RASTER (Cropped to Aargau)
# =============================================================================
cat("── 6. Building resistance raster ──\n")

# --- Extract model coefficients ----------------------------------------------
beta <- coef(m_issf)

# Helper: Safe coefficient lookup
get_beta <- function(name) {
  b <- beta[name]
  if (is.null(b) || is.na(b)) 0 else as.numeric(b)
}

# Helper: Scale a raster using EXACT mean/SD from model training
scale_raster <- function(r, varname) {
  mn <- as.numeric(scale_params[[paste0(varname, "_mean")]])
  sd <- as.numeric(scale_params[[paste0(varname, "_sd")]])
  (r - mn) / sd
}

cat("  Computing linear predictor (η)...\n")

# --- 1. Scale Rasters --------------------------------------------------------
r_forest_dens_sc <- scale_raster(r_forest_dens, "forest_density")
r_dist_infra_sc  <- scale_raster(r_dist_infra, "dist_infra")
r_dist_water_sc  <- scale_raster(r_dist_water, "dist_water")

# --- 2. Calculate Linear Predictor (Eta) -------------------------------------
r_forest_contrib <- r_forest * get_beta("forest_fct1")

eta <- r_forest_contrib +
       (r_forest_dens_sc * get_beta("forest_density_sc")) +
       (r_dist_infra_sc  * get_beta("dist_infra_sc")) +
       (r_dist_water_sc  * get_beta("dist_water_sc"))

# --- 3. Normalise η → Habitat Suitability S ∈ [0, 1] -------------------------
cat("  Normalising to habitat suitability (S)...\n")
eta_min <- global(eta, "min", na.rm = TRUE)[[1]]
eta_max <- global(eta, "max", na.rm = TRUE)[[1]]

S <- (eta - eta_min) / (eta_max - eta_min)

# --- 4. Keeley Exponential Transformation ------------------------------------
cat("  Applying Keeley transformation (C = 4)...\n")
KEELEY_C <- 4
R        <- exp(KEELEY_C * (1 - S))
R_max    <- exp(KEELEY_C)   

# --- 5. Burn-in Absolute Barriers --------------------------------------------
cat("  Burning in absolute barriers...\n")

if (exists("r_highway"))           R[r_highway == 1] <- R_max

# Water barriers
if (exists("r_water_bin")) {
  R[r_water_bin == 1] <- R_max * 0.9
} else if (exists("r_dist_water")) {
  R[r_dist_water == 0] <- R_max * 0.9
}

# Infrastructure barriers
if (exists("r_infra")) {
  R[r_infra == 1] <- R_max
} else if (exists("r_dist_infra")) {
  R[r_dist_infra == 0] <- R_max
}

# --- 6. THE CROP: Mask to Aargau Borders -------------------------------------
cat("  Masking final surfaces to Aargau borders...\n")

# This ensures all pixels outside the canton polygon become NA
S <- mask(S, aargau_vect)
R <- mask(R, aargau_vect)

names(S) <- "habitat_suitability"
names(R) <- "resistance"

# --- 7. Save Outputs ---------------------------------------------------------
cat(sprintf("  Final Resistance range: [%.2f, %.2f]\n",
            global(R, "min", na.rm = TRUE)[[1]],
            global(R, "max", na.rm = TRUE)[[1]]))

writeRaster(S, file.path(OUT_DIR, "Habitat_Suitability_Wildboar_Aargau.tif"), overwrite = TRUE)
writeRaster(R, file.path(OUT_DIR, "Resistance_Keeley_Wildboar_Aargau_25m.tif"), overwrite = TRUE)

cat("  Resistance rasters saved and cropped successfully.\n")

── 6. Building resistance raster ──
  Computing linear predictor (η)...
  Normalising to habitat suitability (S)...
  Applying Keeley transformation (C = 4)...
  Burning in absolute barriers...
  Masking final surfaces to Aargau borders...
  Final Resistance range: [1.00, 54.60]
  Resistance rasters saved and cropped successfully.


In [68]:
cat("── 5. Fitting iSSF on IN-PATCH steps ──\n")

# --- 1. Prepare clean in-patch step data -------------------------------------
in_patch_clean <- in_patch %>%
  filter(
    !is.na(sl_),
    !is.na(ta_),
    sl_ > 0           
  )

cat(sprintf("  In-patch steps for iSSF : %d\n", nrow(in_patch_clean)))

class(in_patch_clean) <- c("steps_xyt", "steps_xy", "data.frame")

# --- 2. Generate random control steps ----------------------------------------
issf_patch_data <- in_patch_clean %>%
  random_steps(n_control = N_RANDOM_STEPS) %>%
  mutate(
    log_sl = log(sl_ + 1),   
    cos_ta = cos(ta_)        
  )

cat(sprintf("  Total rows (observed + random): %d\n", nrow(issf_patch_data)))

# --- 3. Extract spatial covariates at step endpoints -------------------------
cat("  Extracting spatial covariates at step endpoints...\n")

pts_patch <- terra::vect(issf_patch_data, geom = c("x2_", "y2_"), crs = "EPSG:21781")
pts_patch_matched <- terra::project(pts_patch, terra::crs(cov_stack))
ext_vals_patch <- terra::extract(cov_stack, pts_patch_matched)

issf_patch_data <- cbind(issf_patch_data, ext_vals_patch[, -1])   

cov_names <- names(cov_stack)
issf_patch_data <- issf_patch_data %>%
  filter(complete.cases(across(all_of(cov_names)), log_sl, cos_ta))

cat(sprintf("  Valid steps after extraction: %d\n", nrow(issf_patch_data)))

if (nrow(issf_patch_data) == 0) stop("No valid steps. Check CRS alignment!")

# --- 4. Validate and standardise continuous covariates -----------------------
cat("  Checking and scaling landscape variables...\n")

expected_covs <- c("forest_density", "dist_forest_edge", "slope", "tri",
                   "dist_water", "dist_infra")

available_covs <- intersect(expected_covs, names(issf_patch_data))
valid_covs_patch <- c()

for (cov in available_covs) {
  cov_sd <- sd(issf_patch_data[[cov]], na.rm = TRUE)
  if (is.na(cov_sd) || cov_sd == 0) {
    cat(sprintf("  -> WARNING: Raster '%s' is ignored!\n", cov))
  } else {
    valid_covs_patch <- c(valid_covs_patch, cov)
  }
}

# SAVE patch-specific scaling parameters
scale_params_patch <- issf_patch_data %>%
  summarise(across(all_of(valid_covs_patch),
                   list(mean = ~ mean(.x, na.rm = TRUE),
                        sd   = ~ sd(.x,   na.rm = TRUE))))

issf_patch_data <- issf_patch_data %>%
  mutate(across(all_of(valid_covs_patch),
                ~ (.x - mean(.x, na.rm = TRUE)) / sd(.x, na.rm = TRUE),
                .names = "{.col}_sc"))

issf_patch_data$forest_fct <- as.factor(issf_patch_data$forest)

# --- 5. Build formula dynamically --------------------------------------------
scaled_covs_patch <- paste0(valid_covs_patch, "_sc")
form_terms_patch  <- c("forest_fct", "log_sl", "cos_ta", scaled_covs_patch, "strata(step_id_)")

frm_patch <- as.formula(paste("case_ ~", paste(form_terms_patch, collapse = " + ")))

issf_patch_model_data <- issf_patch_data %>% 
  tidyr::drop_na(all_of(c("case_", "forest_fct", "log_sl", "cos_ta", scaled_covs_patch)))

# --- 6. Fit conditional logistic regression ----------------------------------
cat("  Fitting clogit (In-Patch iSSF Model)...\n")

m_issf_patch <- survival::clogit(frm_patch, data = issf_patch_model_data, method = "efron")

cat("  In-Patch iSSF Model SUCCESSFULLY fitted!\n")
print(summary(m_issf_patch))

# --- 7. Plot and save results ------------------------------------------------
coef_df_patch <- broom::tidy(m_issf_patch, conf.int = TRUE) %>%
  filter(!grepl("log_sl|cos_ta|step_id", term)) %>%
  arrange(desc(abs(estimate)))

write.csv(coef_df_patch, file.path(OUT_DIR, "issf_PATCH_coefficients.csv"), row.names = FALSE)

p_coef_patch <- ggplot(coef_df_patch,
                 aes(x = estimate, y = reorder(term, estimate),
                     xmin = conf.low, xmax = conf.high)) +
  geom_vline(xintercept = 0, linetype = "dashed", colour = "grey60") +
  geom_pointrange(colour = "#E69F00", size = 0.7, linewidth = 0.9) + # Orange for patch
  labs(x = "Selection coefficient β (positive = preferred for resting/foraging)",
       y = NULL,
       title = "iSSF — Wild Boar In-Patch Habitat Selection",
       subtitle = "Canton Aargau | Only in-patch steps") +
  theme_minimal(base_size = 11)

ggsave(file.path(OUT_DIR, "issf_PATCH_coefficients.png"), p_coef_patch,
       width = 8, height = 6, dpi = 200, bg = "white")
cat("  Saved: issf_PATCH_coefficients.png\n")





# =============================================================================
# STEP 6: CONVERT COEFFICIENTS → PATCH SUITABILITY RASTER
# =============================================================================
cat("── 6. Building Patch Suitability raster ──\n")

beta_patch <- coef(m_issf_patch)

get_beta_patch <- function(name) {
  b <- beta_patch[name]
  if (is.null(b) || is.na(b)) 0 else as.numeric(b)
}

scale_raster_patch <- function(r, varname) {
  mn <- as.numeric(scale_params_patch[[paste0(varname, "_mean")]])
  sd <- as.numeric(scale_params_patch[[paste0(varname, "_sd")]])
  (r - mn) / sd
}

cat("  Computing linear predictor (η) for resting habitats...\n")

r_forest_dens_sc_p  <- scale_raster_patch(r_forest_dens, "forest_density")
r_dist_infra_sc_p   <- scale_raster_patch(r_dist_infra, "dist_infra")
r_dist_water_sc_p   <- scale_raster_patch(r_dist_water, "dist_water")
r_slope_sc_p        <- scale_raster_patch(r_slope, "slope")
r_tri_sc_p          <- scale_raster_patch(r_tri, "tri")

# Calculate Eta (Comment out any term here that had p > 0.05 in Step 5!)
eta_patch <- (r_forest * get_beta_patch("forest_fct1")) +
             (r_forest_dens_sc_p * get_beta_patch("forest_density_sc")) +
             (r_dist_infra_sc_p  * get_beta_patch("dist_infra_sc")) +
             (r_dist_water_sc_p  * get_beta_patch("dist_water_sc")) +
             (r_slope_sc_p       * get_beta_patch("slope_sc")) +
             (r_tri_sc_p         * get_beta_patch("tri_sc"))

# Normalise to Habitat Suitability S ∈ [0, 1]
eta_p_min <- global(eta_patch, "min", na.rm = TRUE)[[1]]
eta_p_max <- global(eta_patch, "max", na.rm = TRUE)[[1]]

S_patch <- (eta_patch - eta_p_min) / (eta_p_max - eta_p_min)
S_patch <- mask(S_patch, aargau_vect)
names(S_patch) <- "patch_suitability"

writeRaster(S_patch, file.path(OUT_DIR, "Habitat_Suitability_PATCH_Aargau.tif"), overwrite = TRUE)

# =============================================================================
# STEP 6b: EXTRACT CORE RESTING HABITATS (Nodes)
# =============================================================================
cat("── 6b. Extracting core resting patches ──\n")

SUITABILITY_THRESHOLD <- 0.50 
MIN_AREA_HA           <- 25   

r_core_bin <- ifel(S_patch >= SUITABILITY_THRESHOLD, 1, NA)
r_patches  <- patches(r_core_bin, directions = 8)

patch_stats <- freq(r_patches)
PIXEL_HA    <- (25 * 25) / 10000
patch_stats$area_ha <- patch_stats$count * PIXEL_HA

valid_patch_ids <- patch_stats$value[patch_stats$area_ha >= MIN_AREA_HA]
r_core_final <- ifel(r_patches %in% valid_patch_ids, 1, NA)
names(r_core_final) <- "core_resting_habitat"

writeRaster(r_core_final, file.path(OUT_DIR, "Core_Resting_Habitats_Aargau.tif"), overwrite = TRUE)
cat(sprintf("  Identified %d core resting patches!\n", length(valid_patch_ids)))

── 5. Fitting iSSF on IN-PATCH steps ──
  In-patch steps for iSSF : 127002
  Total rows (observed + random): 1397011
  Extracting spatial covariates at step endpoints...
  Valid steps after extraction: 1388319
  Checking and scaling landscape variables...
  -> WARNING: Raster 'dist_forest_edge' is ignored!
  Fitting clogit (In-Patch iSSF Model)...
  In-Patch iSSF Model SUCCESSFULLY fitted!
Call:
coxph(formula = Surv(rep(1, 1388319L), case_) ~ forest_fct + 
    log_sl + cos_ta + forest_density_sc + slope_sc + tri_sc + 
    dist_water_sc + dist_infra_sc + strata(step_id_), data = issf_patch_model_data, 
    method = "efron")

  n= 1388319, number of events= 126214 

                       coef exp(coef)  se(coef)        z Pr(>|z|)    
forest_fct1       -0.356927  0.699823  0.023548  -15.157  < 2e-16 ***
log_sl            -0.043359  0.957567  0.002519  -17.212  < 2e-16 ***
cos_ta            -0.707018  0.493113  0.004284 -165.049  < 2e-16 ***
forest_density_sc  0.214972  1.239827  0.012803

In [69]:
library(terra)
library(sf)

cat("── 6c. Converting and saving Core Habitats as Shapefile ──\n")

# 1. Convert the Raster (pixels) to a Vector (polygons)
# na.rm = TRUE is crucial here: it ensures R only draws polygons around 
# the actual habitat (1) and ignores the empty background (NA).
# dissolve = TRUE merges adjacent pixels into single, large contiguous polygons.
cat("  Vectorising raster pixels to polygons...\n")
core_vect <- as.polygons(r_core_final, dissolve = TRUE, na.rm = TRUE)

# 2. Convert the terra vector object to an 'sf' object
# 'sf' (Simple Features) is the absolute gold standard for vector data in R
core_sf <- st_as_sf(core_vect)

# Optional: Add a clean ID and Area column to your shapefile's attribute table
core_sf$Patch_ID <- 1:nrow(core_sf)
core_sf$Area_HA  <- as.numeric(st_area(core_sf)) / 10000 

# 3. Save as a .shp file
# delete_layer = TRUE allows you to overwrite the file if you run the script again
cat("  Exporting Shapefile...\n")
shp_path <- file.path(OUT_DIR, "Core_Resting_Habitats_Aargau.shp")

st_write(core_sf, 
         dsn = shp_path, 
         driver = "ESRI Shapefile", 
         delete_layer = TRUE, 
         quiet = TRUE)

cat("  Success: Core habitats saved as Shapefile (.shp)!\n")

── 6c. Converting and saving Core Habitats as Shapefile ──
  Vectorising raster pixels to polygons...
  Exporting Shapefile...
  Success: Core habitats saved as Shapefile (.shp)!


## Step 7 — Diagnostic Plots & Summary

In [67]:
# =============================================================================
# STEP 7: DIAGNOSTIC PLOTS & SUMMARY
# =============================================================================

cat("── 7. Diagnostic plots ──\n")

# --- 1. Load data to prevent "object not found" errors -----------------------
cat("  Loading classified telemetry data from CSV...\n")

# Safely establish file paths
patch_file  <- file.path(OUT_DIR, "WildBoar_InPatch_Steps.csv")
matrix_file <- file.path(OUT_DIR, "WildBoar_InMatrix_Steps.csv")

# Verify files exist before loading to prevent harsh crashes
if (file.exists(patch_file) && file.exists(matrix_file)) {
  in_patch  <- read.csv(patch_file)
  in_matrix <- read.csv(matrix_file)
  steps_classified <- bind_rows(in_patch, in_matrix)
} else {
  stop("CSV files from Step 4 not found. Please ensure Step 4 completed successfully.")
}

# --- Plot 1: Sample individual classification track --------------------------
cat("  Generating classification sample track plot...\n")

# Extract the first individual ID to use as a plotting example
sample_id    <- unique(steps_classified$id)[1]
sample_steps <- steps_classified %>% filter(id == sample_id)

p_track <- ggplot(sample_steps, aes(x = x1_, y = y1_, colour = state_label)) +
  geom_path(colour = "grey85", linewidth = 0.3) +
  geom_point(size = 1.0, alpha = 0.7) +
  scale_colour_manual(
    values = c("in-patch" = "#E69F00", "in-matrix" = "#0072B2"),
    name = "State"
  ) +
  coord_equal() +
  labs(
    title = paste("Rule-based classification —", sample_id),
    x = "Easting (LV03)", y = "Northing (LV03)"
  ) +
  theme_minimal(base_size = 11)

# bg = "white" ensures the text remains visible in dark-mode viewers
ggsave(file.path(OUT_DIR, "classification_sample_track.png"), p_track,
       width = 8, height = 7, dpi = 200, bg = "white")
cat("  Saved: classification_sample_track.png\n")

# --- Plot 2: Suitability and Resistance maps side-by-side --------------------
cat("  Generating suitability and resistance maps...\n")

if (exists("S") && exists("R")) {
  png(file.path(OUT_DIR, "suitability_resistance_maps.png"),
      width = 2400, height = 1100, res = 200)
  
  par(mfrow = c(1, 2)) # Create a layout with 1 row and 2 columns
  
  # Plot Suitability (Green)
  plot(S, main = "Habitat Suitability (0–1)\nHigh = preferred during transit",
       col = hcl.colors(50, "Greens 3"), axes = FALSE, plg = list(shrink = 0.9))
  
  # Plot Resistance (Orange/Dark)
  plot(R, main = "Resistance (Keeley C=4)\nLow = permeable",
       col = rev(hcl.colors(50, "Inferno")), axes = FALSE, plg = list(shrink = 0.9))
  
  dev.off()
  cat("  Saved: suitability_resistance_maps.png\n")
} else {
  cat("  Warning: Raster objects 'S' and 'R' not found. Skipping map plot.\n")
}

# --- Plot 3: Resistance value histogram --------------------------------------
cat("  Generating resistance histogram...\n")

if (exists("R") && exists("R_max")) {
  png(file.path(OUT_DIR, "resistance_histogram.png"),
      width = 1200, height = 800, res = 200)
  
  # Extract purely numeric values, dropping NAs to prevent hist() warnings
  r_vals <- na.omit(as.numeric(values(R)))
  
  hist(r_vals,
       breaks = 60, col = "#D85A30", border = "white",
       xlab = "Resistance value", ylab = "Pixel count",
       main = "Distribution of Resistance Values")
  
  # Draw a vertical line showing the absolute barrier threshold
  abline(v = R_max, col = "black", lty = 2, lwd = 1.5)
  text(R_max * 0.96, par("usr")[4] * 0.88,
       "Absolute barrier\n(Infra / Water)", 
       adj = 1, cex = 0.75)
  
  dev.off()
  cat("  Saved: resistance_histogram.png\n")
} else {
  cat("  Warning: Raster 'R' or 'R_max' not found. Skipping histogram.\n")
}

# --- Summary -----------------------------------------------------------------
cat("\n══════════════════════════════════════════════════════\n")
cat("  PIPELINE COMPLETE - SUCCESS!\n")
cat("══════════════════════════════════════════════════════\n\n")
cat("Output files (checked in", OUT_DIR, "):\n")

# List of files we expect to see at the end of the entire workflow
outputs <- c(
  "WildBoar_InPatch_Steps.csv",          
  "WildBoar_InMatrix_Steps.csv",
  "issf_coefficients.csv",
  "issf_coefficients.png",
  "classification_sample_track.png",
  "suitability_resistance_maps.png",
  "resistance_histogram.png",
  "Habitat_Suitability_Wildboar_Aargau.tif",
  "Resistance_Keeley_Wildboar_Aargau_25m.tif"
)

# Verify each file actually exists on the hard drive
for (f in outputs) {
  if (file.exists(file.path(OUT_DIR, f))) {
    cat(sprintf("  ✓ %s\n", f))
  } else {
    cat(sprintf("  ✗ %s (Missing!)\n", f))
  }
}

cat("\nModel Drivers (Landscape Variables Used in Final Map):\n")
cat("  · Forest Cover & Density\n")
cat("  · Distance to Infrastructure\n")
cat("  · Distance to Water\n")
cat("  · Water & Highways (Burned in as Absolute Barriers)\n")

cat("\nReady for: Circuitscape / least-cost path analysis.\n")

── 7. Diagnostic plots ──
  Loading classified telemetry data from CSV...
  Generating classification sample track plot...
  Saved: classification_sample_track.png
  Generating suitability and resistance maps...
  Saved: suitability_resistance_maps.png
  Generating resistance histogram...
  Saved: resistance_histogram.png

══════════════════════════════════════════════════════
  PIPELINE COMPLETE - SUCCESS!
══════════════════════════════════════════════════════

Output files (checked in ../data/processed/ ):
  ✓ WildBoar_InPatch_Steps.csv
  ✓ WildBoar_InMatrix_Steps.csv
  ✓ issf_coefficients.csv
  ✓ issf_coefficients.png
  ✓ classification_sample_track.png
  ✓ suitability_resistance_maps.png
  ✓ resistance_histogram.png
  ✓ Habitat_Suitability_Wildboar_Aargau.tif
  ✓ Resistance_Keeley_Wildboar_Aargau_25m.tif

Model Drivers (Landscape Variables Used in Final Map):
  · Forest Cover & Density
  · Distance to Infrastructure
  · Distance to Water
  · Water & Highways (Burned in as Absolute 